# 多手法・区切りよく提出（doc 18 §2.1.2 の方針）

**方針**: ベースライン改善 6 で「5本目＋重み」は微小帯だったため、**NMF を主軸に 2 本**、**別モデルはオプションで各 1 本**にし、ファイル名で中身が分かるようにする。

| ブロック | 内容 | 提出ファイル（例） |
|----------|------|---------------------|
| **§2 NMF（メイン）** | NMF 追加 LGB を 5 本目に 8% と 12% | submission_20patterns_nmf_08.csv, _nmf_12.csv |
| **§3 別モデル（オプション）** | 重いLGB / CatBoost / XGB / 浅いLGB を各 1 本・5本目 10% | submission_20patterns_heavy_lgb.csv, _catboost.csv, _xgb.csv, _shallow_lgb.csv |

- **NMF だけ回す**: §2 まで実行 → **2 本**。
- **別モデルも試す**: §3 のフラグを True にして実行 → 最大 **6 本**。
- TF-IDF SVD はベースライン改善 6 で試済みのため、このノートでは **NMF のみ** 特徴追加として扱う。

## 1. セットアップ（既存 4 本 ＋ 土台データ）

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from lib.improvement_candidates import get_setup, get_bpr_base
from lib.two_hop import add_2hop_features, TWO_HOP_REVIEW_COUNT
from lib.submission import verify_submission, blend_n_submissions, blend_two_submissions, save_submission

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)
test_ids = ctx.test["ID"].values

path_count1 = ctx.submissions_dir / "submission_2hop_bpr64_count1.csv"
path_bpr128 = ctx.submissions_dir / "submission_2hop_bpr128_only.csv"
path_stacking = ctx.submissions_dir / "submission_improvement_03_stacking.csv"
path_4th = None
for cand in ["submission_similar_movies_reviewed.csv", "submission_improvement_05_scale_pos_weight.csv", "submission_blend_bpr64_count1_bpr128.csv"]:
    p = ctx.submissions_dir / cand
    if p.exists():
        path_4th = p
        break
if path_4th is None:
    blend_two_submissions(path_count1, path_bpr128, ctx.submissions_dir / "submission_blend_bpr64_count1_bpr128.csv", weight_a=0.65, test_ids=test_ids)
    path_4th = ctx.submissions_dir / "submission_blend_bpr64_count1_bpr128.csv"

train_c1, test_c1, feats_c1 = get_bpr_base(ctx, factors=64)
add_2hop_features(train_c1, test_c1, columns=[TWO_HOP_REVIEW_COUNT])
feats_c1 = feats_c1 + [TWO_HOP_REVIEW_COUNT]

def _to_cat(X_tr, X_te, feats):
    for col in feats:
        if not pd.api.types.is_numeric_dtype(X_tr[col]) and not pd.api.types.is_categorical_dtype(X_tr[col]):
            X_tr[col] = X_tr[col].astype("category")
            X_te[col] = X_te[col].astype("category")

print("  4本・土台 OK")

## 2. NMF だけ（メイン）

NMF 追加 LGB を 1 本作り、5 本目 8% と 12% でブレンド → **2 本**出力。

In [ ]:
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.preprocessing import MaxAbsScaler

sub_dir = ctx.submissions_dir
paths_4 = [path_count1, path_bpr128, path_stacking, path_4th]
STACKING_W = 0.45
RATIO_4_4_3 = [4/11, 4/11, 3/11]

# --- NMF のみ（特徴が違うので主軸） ---
text_tr = train_c1["movie_info"].astype(str).replace("nan", "")
text_te = test_c1["movie_info"].astype(str).replace("nan", "")
tfidf = TfidfVectorizer(max_features=200, min_df=2, max_df=0.95, ngram_range=(1, 2), sublinear_tf=True)
n_nmf = 12
X_tf = tfidf.fit_transform(text_tr).toarray()
X_tf_te = tfidf.transform(text_te).toarray()
scaler = MaxAbsScaler()
X_tf = np.clip(scaler.fit_transform(X_tf), 0, None)
X_tf_te = np.clip(scaler.transform(X_tf_te), 0, None)
nmf = NMF(n_components=n_nmf, random_state=42, max_iter=200)
M_nmf_tr = nmf.fit_transform(X_tf)
M_nmf_te = nmf.transform(X_tf_te)
nmf_cols = [f"nmf_{i}" for i in range(n_nmf)]
for i, c in enumerate(nmf_cols):
    train_c1[c] = M_nmf_tr[:, i]
    test_c1[c] = M_nmf_te[:, i]
feats_nmf = feats_c1 + nmf_cols
X_tr_b = train_c1[feats_nmf].copy()
X_te_b = test_c1[feats_nmf].copy()
_to_cat(X_tr_b, X_te_b, feats_nmf)
m_nmf = lgb.LGBMClassifier(**ctx.lgb_params)
m_nmf.fit(X_tr_b, ctx.y)
path_nmf = sub_dir / "submission_20patterns_new_nmf.csv"
save_submission(ctx.test, m_nmf.predict_proba(X_te_b)[:, 1], path_nmf, sanitize=True)
print("  NMF 追加 LGB →", path_nmf.name)

# 5 本目 8% と 12% で 2 本出力
for w in [0.08, 0.12]:
    rest = 0.55 - w
    ws = [rest*RATIO_4_4_3[0], rest*RATIO_4_4_3[1], STACKING_W, rest*RATIO_4_4_3[2], w]
    out = sub_dir / f"submission_20patterns_nmf_{int(w*100):02d}.csv"
    r = blend_n_submissions(paths_4 + [path_nmf], ws, out, test_ids=test_ids)
    print(f"  5本目={w:.0%} → {out.name}  ({r.get('message', '')})")
print("  §2 完了: 2 本")

## 3. 別モデル（オプション）

`RUN_OTHER_MODELS = True` のとき、重いLGB / CatBoost / XGB / 浅いLGB を各 1 本作り、5 本目 10% でブレンド → 最大 **4 本**。False ならスキップ。

In [ ]:
RUN_OTHER_MODELS = True  # False にすると §3 スキップ（NMF 2 本だけ）

if not RUN_OTHER_MODELS:
    print("  §3 スキップ（RUN_OTHER_MODELS=False）")
else:
    # 同じ特徴（feats_c1）のみ
    X_tr_c = train_c1[feats_c1].copy()
    X_te_c = test_c1[feats_c1].copy()
    _to_cat(X_tr_c, X_te_c, feats_c1)
    w5 = 0.10  # 5 本目 10% で 1 本ずつ
    rest = 0.55 - w5
    ws = [rest*RATIO_4_4_3[0], rest*RATIO_4_4_3[1], STACKING_W, rest*RATIO_4_4_3[2], w5]

    # 重い LGB
    path_heavy = sub_dir / "submission_20patterns_new_heavy_lgb.csv"
    m = lgb.LGBMClassifier(**{**ctx.lgb_params, "max_depth": 12, "num_leaves": 128, "min_child_samples": 30})
    m.fit(X_tr_c, ctx.y)
    save_submission(ctx.test, m.predict_proba(X_te_c)[:, 1], path_heavy, sanitize=True)
    r = blend_n_submissions(paths_4 + [path_heavy], ws, sub_dir / "submission_20patterns_heavy_lgb.csv", test_ids=test_ids)
    print(f"  重い LGB → submission_20patterns_heavy_lgb.csv  ({r.get('message', '')})")

    # CatBoost
    try:
        import catboost as cb
        cat_cols = [c for c in feats_c1 if not pd.api.types.is_numeric_dtype(X_tr_c[c])]
        m = cb.CatBoostClassifier(iterations=800, learning_rate=0.05, verbose=False, random_seed=42, early_stopping_rounds=30)
        m.fit(X_tr_c, ctx.y, cat_features=cat_cols if cat_cols else None)
        path_cb = sub_dir / "submission_20patterns_new_catboost.csv"
        save_submission(ctx.test, m.predict_proba(X_te_c)[:, 1], path_cb, sanitize=True)
        r = blend_n_submissions(paths_4 + [path_cb], ws, sub_dir / "submission_20patterns_catboost.csv", test_ids=test_ids)
        print(f"  CatBoost → submission_20patterns_catboost.csv  ({r.get('message', '')})")
    except Exception as e:
        print("  CatBoost スキップ", str(e)[:50])

    # XGBoost
    try:
        import xgboost as xgb
        X_tr_e = X_tr_c.copy()
        X_te_e = X_te_c.copy()
        for c in X_tr_e.columns:
            if X_tr_e[c].dtype.name == "category":
                cats = X_tr_c[c].cat.categories
                X_tr_e[c] = X_tr_c[c].cat.codes
                X_te_e[c] = pd.Categorical(X_te_c[c].astype(object), categories=cats).codes
        m = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, random_state=42, use_label_encoder=False, eval_metric="logloss")
        m.fit(X_tr_e, ctx.y)
        path_xgb = sub_dir / "submission_20patterns_new_xgb.csv"
        save_submission(ctx.test, m.predict_proba(X_te_e)[:, 1], path_xgb, sanitize=True)
        r = blend_n_submissions(paths_4 + [path_xgb], ws, sub_dir / "submission_20patterns_xgb.csv", test_ids=test_ids)
        print(f"  XGBoost → submission_20patterns_xgb.csv  ({r.get('message', '')})")
    except Exception as e:
        print("  XGBoost スキップ", str(e)[:50])

    # 浅い LGB
    m = lgb.LGBMClassifier(**{**ctx.lgb_params, "max_depth": 4, "num_leaves": 16, "min_child_samples": 80, "reg_alpha": 0.5, "reg_lambda": 0.5})
    m.fit(X_tr_c, ctx.y)
    path_shallow = sub_dir / "submission_20patterns_new_shallow_lgb.csv"
    save_submission(ctx.test, m.predict_proba(X_te_c)[:, 1], path_shallow, sanitize=True)
    r = blend_n_submissions(paths_4 + [path_shallow], ws, sub_dir / "submission_20patterns_shallow_lgb.csv", test_ids=test_ids)
    print(f"  浅い LGB → submission_20patterns_shallow_lgb.csv  ({r.get('message', '')})")
    print("  §3 完了: 最大 4 本")

## 4. 提出ファイル一覧

In [ ]:
names = [
    "submission_20patterns_nmf_08.csv", "submission_20patterns_nmf_12.csv",
    "submission_20patterns_heavy_lgb.csv", "submission_20patterns_catboost.csv",
    "submission_20patterns_xgb.csv", "submission_20patterns_shallow_lgb.csv",
]
for name in names:
    p = ctx.submissions_dir / name
    if p.exists():
        v = verify_submission(p, ctx.test)
        print(f"  {p.name}  ({v.get('message', '')})")
    else:
        print(f"  {name}  (未生成)")